# Amazon ML Challenge 2026: Business Entity Resolution Pipeline
### End-to-End Execution & Cloud Setup Guide for AWS SageMaker (`ml.c5.18xlarge`)

**Team Name:** unwanted  
**Target Competition Metric:** Official Macro $F_{0.5} \ge 0.9000+$  

---
### ⚙️ SageMaker Cloud Infrastructure & Space Provisioning
To reproduce this solution without disk or memory constraints, provision the environment as follows:

1. **Instance Type:**
   - **Recommended:** `ml.c5.18xlarge` (72 vCPUs, 144 GB RAM, network-optimized).
   - **Alternative:** `ml.m5.16xlarge` (64 vCPUs, 256 GB RAM) or `ml.c5.9xlarge` (36 vCPUs, 72 GB RAM).

2. **EBS Storage Allocation (Crucial — Avoid Disk-Full Failures):**
   - **Required Volume Size:** At least **150 GB to 200 GB** (EBS gp3, 3,000+ IOPS, 125+ MB/s).
   - **Why?** Uncompressed TSV datasets require ~15 GB, the SQLite B-Tree search index (`index.db`) requires ~14 GB, test index (`index_test.db`) requires ~12 GB, and feature vectors require ~10 GB.
   - **How to create/expand space:**
     - *When creating instance:* In AWS SageMaker Console $\rightarrow$ *Notebook instances* $\rightarrow$ *Create notebook instance* $\rightarrow$ Set **Volume size in GB = 200**.
     - *On existing instance:* Stop notebook instance $\rightarrow$ Select *Update* $\rightarrow$ Increase volume size to 200 GB $\rightarrow$ Start instance.
     - Verify free space in terminal with `df -h /`.

3. **Zero-RAM Disk-Indexed Architecture:**
   - All 10.1M records in Source 2 and Source 3 are indexed directly into SQLite B-Tree tables on disk.
   - Uses 36 parallel worker processes (safe deadlock-free sweet spot) to maximize CPU utilization without SQLite reader lock contention.
   - Checkpoints are saved after every chunk to `output/train/lgb_model.pkl` and `output/lgb_model.pkl`.
   - Results are automatically validated against the official competition validator (`validate_submission.py`) and packaged into `unwanted_submission.zip`.


--- 
## 1. Environment & Dependency Setup
Creates a Python virtual environment (`.venv`) and installs project dependencies.

In [ ]:
%%bash
if [ -d "$HOME/Amazon-ML-Hackathon-2026" ]; then
    cd "$HOME/Amazon-ML-Hackathon-2026"
elif [ -d "$HOME/SageMaker/Amazon-ML-Hackathon-2026" ]; then
    cd "$HOME/SageMaker/Amazon-ML-Hackathon-2026"
elif [ -d "$HOME/SageMaker/ml" ]; then
    cd "$HOME/SageMaker/ml"
fi

# Create .venv if not already created
if [ ! -d ".venv" ]; then
    python3 -m venv .venv
    echo "Virtual environment .venv created."
fi

source .venv/bin/activate
pip install --upgrade pip
pip install pandas numpy lightgbm pyarrow scikit-learn boto3 matplotlib seaborn xgboost catboost


--- 
## 2. Sync Latest Code from GitHub

In [ ]:
%%bash
if [ -d "$HOME/Amazon-ML-Hackathon-2026" ]; then
    cd "$HOME/Amazon-ML-Hackathon-2026"
elif [ -d "$HOME/SageMaker/Amazon-ML-Hackathon-2026" ]; then
    cd "$HOME/SageMaker/Amazon-ML-Hackathon-2026"
elif [ -d "$HOME/SageMaker/ml" ]; then
    cd "$HOME/SageMaker/ml"
fi
git stash
git pull origin main


--- 
## 3. Verify Dataset Files
Ensures all 7 TSV dataset files are present in `dataset/train/` and `dataset/test/`.

In [ ]:
%%bash
if [ -d "$HOME/Amazon-ML-Hackathon-2026" ]; then
    cd "$HOME/Amazon-ML-Hackathon-2026"
elif [ -d "$HOME/SageMaker/Amazon-ML-Hackathon-2026" ]; then
    cd "$HOME/SageMaker/Amazon-ML-Hackathon-2026"
elif [ -d "$HOME/SageMaker/ml" ]; then
    cd "$HOME/SageMaker/ml"
fi
echo "=== TRAIN DATASET ==="
ls -lh dataset/train/

echo ""
echo "=== TEST DATASET ==="
ls -lh dataset/test/

echo ""
echo "=== SQLITE INDEX DATABASES ==="
ls -lh index*.db 2>/dev/null || true


---
## 4. Run Multi-Chunk Continual Training (`--split train`)
Runs continual ensemble training across 1,600,000 Source 1 records (~70% of dataset):
- **8 sequential chunks** of 200,000 entities each (Zero-RAM leak-proof streaming).
- **68 parallel workers** fully saturating the 72 vCPUs on `ml.c5.18xlarge`.
- **80 candidates** retrieved per record to maximize Stage 1 candidate recall ($F_{0.5}$ ceiling > 0.95).
- **Fixed holdout benchmark** (5,000 entities) evaluated after each chunk to monitor true Macro $F_{0.5}$.
- **Checkpoints saved** after every chunk to `output/train/lgb_model.pkl` and `output/lgb_model.pkl`.

In [ ]:
%%bash
if [ -d "$HOME/Amazon-ML-Hackathon-2026" ]; then
    cd "$HOME/Amazon-ML-Hackathon-2026"
elif [ -d "$HOME/SageMaker/Amazon-ML-Hackathon-2026" ]; then
    cd "$HOME/SageMaker/Amazon-ML-Hackathon-2026"
elif [ -d "$HOME/SageMaker/ml" ]; then
    cd "$HOME/SageMaker/ml"
fi
source .venv/bin/activate

# ==============================================================================
# MULTI-CHUNK CONTINUAL TRAINING ON C5.18XLARGE (72 vCPUs)
# Processes 8 sequential chunks of 200,000 Source 1 entities = 1,600,000 records (~70% of dataset)
# - num-workers: 68 parallel processes (saturates 72 vCPUs safely without freeze)
# - max-candidates: 80 (breaks Stage 1 candidate recall bottleneck to maximize F0.5)
# - trees-per-chunk: 50 (continual LightGBM ensemble expansion)
# - val-size: 5,000 holdout entities (fixed benchmark evaluated after each chunk)
# ==============================================================================
python3 code/src/main.py \
    --split train \
    --num-chunks 4 \
    --chunk-size 40000 \
    --chunk-offset 0 \
    --max-candidates 180 \
    --val-size 5000 \
    --num-workers 36 \
    --trees-per-chunk 100

# NOTE: If training is ever stopped or interrupted, resume from last saved checkpoint with:
# python3 code/src/main.py --split train --num-chunks 8 --chunk-size 200000 --chunk-offset 0 --max-candidates 150 --val-size 5000 --num-workers 68 --trees-per-chunk 50 --continue-training


--- 
## 5. Check Training Artifacts & Model Status

In [ ]:
import os
import glob
import pickle

print("=== Training Artifacts (output/train/) ===")
train_dir = "output/train"
if os.path.exists(train_dir):
    for f in sorted(os.listdir(train_dir)):
        p = os.path.join(train_dir, f)
        size_mb = os.path.getsize(p) / (1024 * 1024)
        print(f"  - {f:35s}: {size_mb:8.2f} MB")
else:
    print("  (No output/train directory yet. Run Section 4 to train.)")

print("
=== Testing Outputs (output/test/) ===")
test_dir = "output/test"
if os.path.exists(test_dir):
    for f in sorted(os.listdir(test_dir)):
        p = os.path.join(test_dir, f)
        size_mb = os.path.getsize(p) / (1024 * 1024)
        print(f"  - {f:35s}: {size_mb:8.2f} MB")
else:
    print("  (No output/test directory yet. Run Section 6 to run test inference.)")

model_path = "output/train/lgb_model.pkl" if os.path.exists("output/train/lgb_model.pkl") else "output/lgb_model.pkl"
if os.path.exists(model_path):
    with open(model_path, "rb") as f:
        m = pickle.load(f)
    print(f"
Trained Model Ready at {model_path}! Optimal Macro F0.5 Threshold: {m['optimal_threshold']:.3f}")


--- 
## 5a. Evaluate Upgraded High-Recall Model on Validation Set
Runs standalone validation audit on the 5,000 holdout entities using upgraded multi-signal conjunction indexing and  with 68 workers (~45 seconds).

In [ ]:
%%bash
if [ -d "$HOME/Amazon-ML-Hackathon-2026" ]; then
    cd "$HOME/Amazon-ML-Hackathon-2026"
elif [ -d "$HOME/SageMaker/Amazon-ML-Hackathon-2026" ]; then
    cd "$HOME/SageMaker/Amazon-ML-Hackathon-2026"
elif [ -d "$HOME/SageMaker/ml" ]; then
    cd "$HOME/SageMaker/ml"
fi
source .venv/bin/activate

# Run fast validation evaluation with upgraded high-recall retrieval
python3 code/src/main.py \
    --split train \
    --val-size 5000 \
    --max-candidates 180 \
    --num-workers 36 \
    --no-cache \
    --eval-only


--- 
## 5b. Validation Confusion Matrix & Metrics Visualization
Loads cached validation pairs and evaluates True/False Positives/Negatives with an annotated heatmap.

In [ ]:
import os
import glob
import pickle
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

model_file = "output/train/lgb_model.pkl" if os.path.exists("output/train/lgb_model.pkl") else "output/lgb_model.pkl"
val_bench_file = "output/train/holdout_val_benchmark.pkl"
val_metrics = None
val_metrics_file = "output/train/val_metrics.pkl"
if os.path.exists(val_metrics_file):
    try:
        with open(val_metrics_file, "rb") as f:
            val_metrics = pickle.load(f)
        optimal_thresh = val_metrics.get("threshold", 0.500)
        print(f"Loaded fresh evaluation metrics from: {val_metrics_file}")
    except Exception as e:
        print(f"Note: Could not read val_metrics.pkl: {e}")
optimal_thresh = 0.500

# 1. Try to load val_metrics directly from model checkpoint
if os.path.exists(model_file):
    try:
        with open(model_file, "rb") as f:
            model_data = pickle.load(f)
        optimal_thresh = model_data.get("optimal_threshold", 0.500)
        val_metrics = model_data.get("val_metrics", None)
        print(f"Loaded model checkpoint from: {model_file}")
    except Exception as e:
        print(f"Note: Could not read val_metrics from model file: {e}")

# 2. If val_metrics not directly in model, evaluate on locked holdout validation benchmark
if (not val_metrics or not val_metrics.get("macro_f05")) and os.path.exists(val_bench_file) and os.path.exists(model_file):
    try:
        print(f"Evaluating model directly on locked holdout benchmark ({val_bench_file})...")
        with open(val_bench_file, "rb") as f:
            bench = pickle.load(f)
        with open(model_file, "rb") as f:
            mdata = pickle.load(f)
        clf = mdata["clf"]
        X_val, y_val = bench["X_val"], bench["y_val"]
        val_probs = clf.predict_proba(X_val)[:, 1]
        val_preds = (val_probs >= optimal_thresh).astype(int)
        cm = confusion_matrix(y_val, val_preds)
        tn, fp, fn, tp = cm.ravel()
        prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        rec = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = (2 * prec * rec) / (prec + rec) if (prec + rec) > 0 else 0.0
        f05 = (1.25 * prec * rec) / (0.25 * prec + rec) if (0.25 * prec + rec) > 0 else 0.0
        val_metrics = {
            "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
            "cand_prec": prec, "cand_rec": rec, "f1": f1, "cand_f05": f05,
            "threshold": optimal_thresh
        }
    except Exception as e:
        print(f"Benchmark evaluation error: {e}")

if not val_metrics:
    print("No validation metrics found yet. Run Section 4 to train.")
else:
    tn = val_metrics.get("tn", 0)
    fp = val_metrics.get("fp", 0)
    fn = val_metrics.get("fn", 0)
    tp = val_metrics.get("tp", 0)
    prec = val_metrics.get("cand_prec", val_metrics.get("prec", 0.0))
    rec = val_metrics.get("cand_rec", val_metrics.get("rec", 0.0))
    pairwise_f1 = val_metrics.get("f1", 0.0)
    pairwise_f05 = val_metrics.get("cand_f05", val_metrics.get("f05", 0.0))
    blocking_rec = val_metrics.get("blocking_recall", None)
    macro_f05 = val_metrics.get("macro_f05", pairwise_f05)
    threshold = val_metrics.get("threshold", optimal_thresh)
    cm = np.array([[tn, fp], [fn, tp]])
    
    print("=" * 65)
    print(f"   VALIDATION AUDIT & CONFUSION MATRIX (Threshold = {threshold:.3f})   ")
    print("=" * 65)
    print(f"                 PREDICTED NON-MATCH       PREDICTED MATCH")
    print(f"ACTUAL NON-MATCH   TN: {tn:10,d}           FP: {fp:10,d}")
    print(f"ACTUAL MATCH       FN: {fn:10,d}           TP: {tp:10,d}")
    print("-" * 65)
    print(f"  True Positives  (TP) : {tp:8,d}  (Correct candidate matches captured)")
    print(f"  False Positives (FP) : {fp:8,d}  (False alarms)")
    print(f"  False Negatives (FN) : {fn:8,d}  (Missed true matches in candidate pool)")
    print(f"  True Negatives  (TN) : {tn:8,d}  (Correct non-matches)")
    print("-" * 65)
    if blocking_rec is not None:
        print(f"  Stage 1 Blocking Recall : {blocking_rec:.2f}%")
    print(f"  Pairwise Precision      : {prec * 100:.2f}%")
    print(f"  Pairwise Recall (cands) : {rec * 100:.2f}%")
    print(f"  Pairwise F0.5 (cands)   : {pairwise_f05:.4f}")
    print(f"  >>> OFFICIAL MACRO F0.5 : {macro_f05:.4f} <<< [COMPETITION METRIC]")
    print("=" * 65)
    
    plt.figure(figsize=(7, 5))
    annot = np.array([[f"{tn:,}\n(TN)", f"{fp:,}\n(FP)"], [f"{fn:,}\n(FN)", f"{tp:,}\n(TP)"]])
    sns.heatmap(cm, annot=annot, fmt="", cmap="Blues", cbar=True,
                xticklabels=["Predicted Non-Match", "Predicted Match"],
                yticklabels=["Actual Non-Match", "Actual Match"])
    plt.title(f"Validation Confusion Matrix (Threshold = {threshold:.3f})", fontsize=13, pad=12)
    plt.ylabel("Ground Truth Label", fontsize=11)
    plt.xlabel("Model Prediction", fontsize=11)
    plt.tight_layout()
    plt.show()


---
## 6. Run Test Split Inference (`--split test`)
Executes high-throughput candidate retrieval and scoring on `test_source1.tsv` (1.73M entities):
- Uses **36 parallel workers** streaming directly from disk index (`index_test.db`).
- Uses **180 candidates** per record matching the training configuration.
- Produces official competition outputs:
  - `output/test/candidate_pairs.tsv`
  - `output/test/matching_results.tsv`


In [ ]:
%%bash
if [ -d "$HOME/Amazon-ML-Hackathon-2026" ]; then
    cd "$HOME/Amazon-ML-Hackathon-2026"
elif [ -d "$HOME/SageMaker/Amazon-ML-Hackathon-2026" ]; then
    cd "$HOME/SageMaker/Amazon-ML-Hackathon-2026"
elif [ -d "$HOME/SageMaker/ml" ]; then
    cd "$HOME/SageMaker/ml"
fi
source .venv/bin/activate

# Full test set inference (1.73M entities) using 36 parallel workers and top-180 candidates
python3 code/src/main.py \
    --split test \
    --sample-size 0 \
    --max-candidates 180 \
    --num-workers 36


--- 
## 7. Official Submission Format Validation

In [ ]:
%%bash
if [ -d "$HOME/Amazon-ML-Hackathon-2026" ]; then
    cd "$HOME/Amazon-ML-Hackathon-2026"
elif [ -d "$HOME/SageMaker/Amazon-ML-Hackathon-2026" ]; then
    cd "$HOME/SageMaker/Amazon-ML-Hackathon-2026"
elif [ -d "$HOME/SageMaker/ml" ]; then
    cd "$HOME/SageMaker/ml"
fi
source .venv/bin/activate

# Run official competition validator on output/test/ files
python3 code/src/validate_submission.py \
    --matching output/test/matching_results.tsv \
    --candidate output/test/candidate_pairs.tsv \
    --test-dir dataset/test/


--- 
## 8. Preview Final Submission Files

In [ ]:
import os
import pandas as pd

cand_file = "output/test/candidate_pairs.tsv" if os.path.exists("output/test/candidate_pairs.tsv") else "output/candidate_pairs.tsv"
if os.path.exists(cand_file):
    df_cand = pd.read_csv(cand_file, sep="	", nrows=10)
    print("=== Candidate Pairs (Top 10 Rows) ===")
    display(df_cand)
else:
    print(f"Candidate file not found at {cand_file}")

print("
=== Matching Results (Top 10 Rows) ===")
match_file = "output/test/matching_results.tsv" if os.path.exists("output/test/matching_results.tsv") else "output/matching_results.tsv"
if os.path.exists(match_file):
    df_match = pd.read_csv(match_file, sep="	", nrows=10)
    display(df_match)
else:
    print(f"Matching file not found at {match_file}")


--- 
## 9. (Optional) Sync Output Files to S3 Bucket

In [ ]:
%%bash
# Sync final outputs to your S3 bucket
aws s3 cp output/test/matching_results.tsv s3://sagemaker-eu-north-1-583664563299/amazon-ml-challenge-2026/output/matching_results.tsv
aws s3 cp output/test/candidate_pairs.tsv s3://sagemaker-eu-north-1-583664563299/amazon-ml-challenge-2026/output/candidate_pairs.tsv
echo "Outputs backed up to S3 bucket!"


---
## 9. Build Final Submission Archive (unwanted_submission.zip)
Packages `output/`, `code/business_entity_resolution/`, and `Documentation_template.md` into `unwanted_submission.zip` matching official competition specifications.

In [ ]:
%%bash
if [ -d "$HOME/Amazon-ML-Hackathon-2026" ]; then
    cd "$HOME/Amazon-ML-Hackathon-2026"
elif [ -d "$HOME/SageMaker/Amazon-ML-Hackathon-2026" ]; then
    cd "$HOME/SageMaker/Amazon-ML-Hackathon-2026"
elif [ -d "$HOME/SageMaker/ml" ]; then
    cd "$HOME/SageMaker/ml"
fi
source .venv/bin/activate

# Automatically sync code, copy outputs, validate, and create unwanted_submission.zip
python3 package_submission.py
